# NGX XGBoost Retraining
**Trains a binary UP/DOWN XGBoost classifier on NGX price data from BigQuery.**

| Setting | Value |
|---|---|
| Training window | Rolling 2 years |
| Validation | Last 60 calendar days (chronological) |
| Label | Next-day close > current × 1.005 → UP |
| Quality gate | ROC-AUC ≥ 0.55, MCC ≥ 0.05, Balanced-Acc ≥ 0.55 |

**After training:** upload the downloaded zip to the backend repo:
- `xgboost_model.pkl` → `models/xgboost_model.pkl`
- `xgb_feature_list.json` → `configs/xgb_feature_list.json`
- `xgb_metrics.json` → `reports/xgb_metrics.json`

In [ ]:
# Cell A — Install dependencies
# Run this cell first; restart the runtime if Colab prompts you to.
!pip install -q xgboost scikit-learn pandas numpy pyarrow joblib google-cloud-bigquery db-dtypes

In [ ]:
# Cell B — Imports + Config
import os
import json
import subprocess
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime, timezone
from sklearn.metrics import (
    roc_auc_score, classification_report,
    matthews_corrcoef, balanced_accuracy_score,
)
from xgboost import XGBClassifier

# BigQuery
BQ_PROJECT = "stock-market-pipeline-496521"
BQ_DATASET = "ngx_market_data"
BQ_TABLE   = "price"

# Training
WINDOW_DAYS     = 730    # rolling 2-year training window
HOLDOUT_DAYS    = 60     # validation holdout (chronological)
MIN_TICKER_ROWS = 60     # skip tickers with fewer rows
UP_THRESHOLD    = 0.005  # next-day return > 0.5% = UP (class 1)

# Quality gate
MIN_AUC    = 0.55
MIN_MCC    = 0.05
MIN_BALACC = 0.55

# XGBoost hyper-parameters
XGB_PARAMS = dict(
    n_estimators          = 1000,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    objective             = "binary:logistic",
    eval_metric           = "auc",
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 50,
)

# Auto-detect GPU
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    XGB_PARAMS["device"] = "cuda"
    print("GPU detected — training on CUDA")
except Exception:
    print("No GPU — training on CPU")
    print("Tip: Runtime → Change runtime type → T4 GPU")

OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Config ready.")

In [ ]:
# Cell C — Authenticate + Load from BigQuery

# ── Colab (default) ──────────────────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()

# ── Kaggle: comment out the two lines above and uncomment this line ───────────
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/kaggle/input/YOUR-KEY-FOLDER/key.json"
# ─────────────────────────────────────────────────────────────────────────────

from google.cloud import bigquery
client = bigquery.Client(project=BQ_PROJECT)

query = f"""
SELECT
    date, ticker,
    pclose, high, low, close, volume, change
FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`
WHERE close  IS NOT NULL
  AND close  > 0
  AND high   > 0
  AND low    > 0
ORDER BY ticker, date
"""

print("Querying BigQuery…")
df_raw = client.query(query).to_dataframe()
df_raw["date"] = pd.to_datetime(df_raw["date"])

print(f"Loaded {len(df_raw):,} rows | {df_raw['ticker'].nunique()} tickers")
print(f"Date range: {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
df_raw.head()

In [ ]:
# Cell D — Clean data
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset=["date", "ticker"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    df = df[
        (df["close"] > 0) &
        (df["high"]  >= df["low"]) &
        (df["high"]  >= df["close"]) &
        (df["low"]   <= df["close"])
    ]
    df["volume"] = (
        df.groupby("ticker")["volume"]
        .transform(lambda x: x.fillna(x.median()).clip(lower=0))
        .fillna(0)
    )
    df["pclose"] = df.groupby("ticker")["pclose"].transform(
        lambda x: x.ffill().bfill()
    )
    return df

df = clean_data(df_raw)
print(f"After cleaning: {len(df):,} rows  ({len(df_raw) - len(df):,} removed)")

In [ ]:
# Cell E — Feature engineering
# Replicates app/services/feature_engineer.py so the model is backend-compatible.

def _rsi(close: pd.Series, w: int = 14) -> pd.Series:
    d    = close.diff()
    gain = d.clip(lower=0).rolling(w, min_periods=max(2, w // 2)).mean()
    loss = (-d.clip(upper=0)).rolling(w, min_periods=max(2, w // 2)).mean()
    rs   = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def _macd(close: pd.Series):
    ema12  = close.ewm(span=12, adjust=False, min_periods=6).mean()
    ema26  = close.ewm(span=26, adjust=False, min_periods=13).mean()
    macd   = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False, min_periods=4).mean()
    return macd, signal, macd - signal

def _bollinger(close: pd.Series, w: int = 20):
    mid   = close.rolling(w, min_periods=5).mean()
    std   = close.rolling(w, min_periods=5).std()
    upper = mid + 2 * std
    lower = mid - 2 * std
    return mid, upper, lower, (upper - lower) / mid.replace(0, np.nan)

def _atr(high, low, close, w: int = 14) -> pd.Series:
    pc = close.shift(1)
    tr = pd.concat([high - low, (high - pc).abs(), (low - pc).abs()], axis=1).max(axis=1)
    return tr.rolling(w, min_periods=max(2, w // 2)).mean()

def _obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    return (np.sign(close.diff()).fillna(0) * volume.fillna(0)).cumsum()

def engineer_ticker(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy().sort_values("date")
    c = g["close"].astype(float)
    h = g["high"].astype(float)
    l = g["low"].astype(float)
    v = g["volume"].astype(float)

    g["daily_return"] = c.pct_change()
    g["log_return"]   = np.log(c.replace(0, np.nan) / c.shift(1).replace(0, np.nan))
    for w in (3, 5, 10, 20):
        g[f"return_{w}d"] = c.pct_change(w)

    g["volatility_20"] = g["daily_return"].rolling(20, min_periods=5).std() * np.sqrt(252)

    for w in (5, 10, 20, 50):
        g[f"ma_{w}"] = c.rolling(w, min_periods=max(2, w // 4)).mean()
    g["ma_20_gap_pct"] = ((c - g["ma_20"]) / g["ma_20"].replace(0, np.nan)) * 100
    g["ma_50_gap_pct"] = ((c - g["ma_50"]) / g["ma_50"].replace(0, np.nan)) * 100

    g["rsi_14"] = _rsi(c, 14)
    g["macd"], g["macd_signal"], g["macd_hist"] = _macd(c)
    g["bb_mid"], g["bb_upper"], g["bb_lower"], g["bb_width"] = _bollinger(c)
    g["atr_14"] = _atr(h, l, c, 14)
    g["obv"]    = _obv(c, v)

    avg_v20 = v.rolling(20, min_periods=5).mean()
    g["volume_ratio_20"] = v / avg_v20.replace(0, np.nan)
    g["volume_change"]   = v.pct_change()

    rh52 = c.rolling(252, min_periods=20).max()
    g["drawdown_52w"]   = ((c / rh52.replace(0, np.nan)) - 1.0) * 100
    g["high_low_pct"]   = ((h - l) / c.replace(0, np.nan)) * 100
    g["close_position"] = (c - l) / (h - l).replace(0, np.nan)

    for lag in (1, 2):
        g[f"lag_close_{lag}"]  = c.shift(lag)
        g[f"lag_return_{lag}"] = g["daily_return"].shift(lag)
    g["lag_volume_1"] = v.shift(1)

    return g

print("Engineering features — takes ~1-2 minutes…")
frames = []
skipped = 0
for ticker, group in df.groupby("ticker"):
    if len(group) >= MIN_TICKER_ROWS:
        frames.append(engineer_ticker(group))
    else:
        skipped += 1

df_feat = pd.concat(frames, ignore_index=True)
df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
print(f"Done: {len(df_feat):,} rows, {df_feat['ticker'].nunique()} tickers  ({skipped} skipped)")

In [ ]:
# Cell F — Labels + chronological train/val split

# 33 features that EXACTLY match backend FALLBACK_FEATURE_COLUMNS
FEATURE_COLS = [
    "daily_return", "log_return",
    "return_3d", "return_5d", "return_10d", "return_20d",
    "volatility_20",
    "ma_5", "ma_10", "ma_20", "ma_50",
    "ma_20_gap_pct", "ma_50_gap_pct",
    "rsi_14", "macd", "macd_signal", "macd_hist",
    "bb_mid", "bb_upper", "bb_lower", "bb_width",
    "atr_14", "obv",
    "volume_ratio_20", "volume_change",
    "drawdown_52w", "high_low_pct", "close_position",
    "lag_close_1", "lag_close_2",
    "lag_return_1", "lag_return_2",
    "lag_volume_1",
]

# Label: next-day return > 0.5% = UP
df_feat = df_feat.sort_values(["ticker", "date"])
df_feat["_next_close"] = df_feat.groupby("ticker")["close"].shift(-1)
df_feat["_fwd_return"] = (
    (df_feat["_next_close"] - df_feat["close"]) /
    df_feat["close"].replace(0, np.nan)
)
df_feat["label"] = (df_feat["_fwd_return"] > UP_THRESHOLD).astype(int)
df_feat = df_feat.dropna(subset=["label", "daily_return", "rsi_14"])

# Rolling 2-year window
latest = df_feat["date"].max()
df_win = df_feat[df_feat["date"] >= latest - pd.Timedelta(days=WINDOW_DAYS)].copy()

# Chronological split — never shuffle financial time series
holdout_start = df_win["date"].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train_df = df_win[df_win["date"] <  holdout_start]
val_df   = df_win[df_win["date"] >= holdout_start]

X_train = train_df[FEATURE_COLS].fillna(0.0)
y_train = train_df["label"].astype(int)
X_val   = val_df[FEATURE_COLS].fillna(0.0)
y_val   = val_df["label"].astype(int)

print(f"Train: {len(X_train):,} rows | Val: {len(X_val):,} rows")
print(f"Label balance — Train UP: {y_train.mean()*100:.1f}% | Val UP: {y_val.mean()*100:.1f}%")

In [ ]:
# Cell G — Train XGBoost
n_down = int((y_train == 0).sum())
n_up   = int(max((y_train == 1).sum(), 1))
scale_pos_weight = max(1.0, n_down / n_up)

print(f"Class balance — DOWN: {n_down:,}  UP: {n_up:,}  ({100*n_up/(n_down+n_up):.1f}% UP)")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
print("\nTraining…")

params = {**XGB_PARAMS, "scale_pos_weight": scale_pos_weight}
model  = XGBClassifier(**params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

print(f"\nBest iteration: {model.best_iteration}")

In [ ]:
# Cell H — Evaluate
y_prob  = model.predict_proba(X_val)[:, 1]
y_pred  = (y_prob >= 0.5).astype(int)

auc     = float(roc_auc_score(y_val, y_prob))
mcc     = float(matthews_corrcoef(y_val, y_pred))
bal_acc = float(balanced_accuracy_score(y_val, y_pred))
passed  = (auc >= MIN_AUC) and (mcc >= MIN_MCC) and (bal_acc >= MIN_BALACC)

print(f"\n{'='*58}")
print(f"  ROC-AUC            {auc:.4f}   gate \u2265 {MIN_AUC}   {'\u2713' if auc >= MIN_AUC else '\u2717'}")
print(f"  MCC                {mcc:.4f}   gate \u2265 {MIN_MCC}   {'\u2713' if mcc >= MIN_MCC else '\u2717'}")
print(f"  Balanced Accuracy  {bal_acc:.4f}   gate \u2265 {MIN_BALACC}   {'\u2713' if bal_acc >= MIN_BALACC else '\u2717'}")
print(f"{'='*58}")
print(f"  Quality gate: {'PASSED \u2713' if passed else 'FAILED \u2717'}")
print(f"{'='*58}")
print(f"\nPredicted UP%: {y_pred.mean()*100:.1f}%  (actual: {y_val.mean()*100:.1f}%)")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["DOWN", "UP"]))

fi = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:")
print(fi.head(15).to_string())

if not passed:
    up_pct = y_train.mean() * 100
    print("\nTroubleshooting:")
    if up_pct < 10:
        print(f"  \u26a0 UP labels only {up_pct:.1f}% of training data.")
        print("    Try lowering UP_THRESHOLD (e.g. 0.001 or 0.0)")
    print("  \u2192 Increase n_estimators to 2000")
    print("  \u2192 Try max_depth=6, learning_rate=0.03")

In [ ]:
# Cell I — Save artifacts
if passed:
    joblib.dump(model, OUTPUT_DIR / "xgboost_model.pkl")

    with open(OUTPUT_DIR / "xgb_feature_list.json", "w") as f:
        json.dump(FEATURE_COLS, f, indent=2)

    metrics_out = {
        "auc_roc":           round(auc, 4),
        "mcc":               round(mcc, 4),
        "balanced_accuracy": round(bal_acc, 4),
        "train_rows":        len(X_train),
        "val_rows":          len(X_val),
        "best_iteration":    int(model.best_iteration),
        "n_features":        len(FEATURE_COLS),
        "features":          FEATURE_COLS,
        "scale_pos_weight":  round(scale_pos_weight, 4),
        "up_threshold":      UP_THRESHOLD,
        "generated_at":      datetime.now(timezone.utc).isoformat(),
    }
    with open(OUTPUT_DIR / "xgb_metrics.json", "w") as f:
        json.dump(metrics_out, f, indent=2)

    print("\u2713 Saved to artifacts/")
    print("  xgboost_model.pkl      \u2192 models/xgboost_model.pkl")
    print("  xgb_feature_list.json  \u2192 configs/xgb_feature_list.json")
    print("  xgb_metrics.json       \u2192 reports/xgb_metrics.json")
else:
    print("\u2717 Model NOT saved. Fix issues in Cell H then re-run Cell G.")

In [ ]:
# Cell J — Zip + Download
if passed:
    import shutil
    shutil.make_archive("ngx_xgboost_artifacts", "zip", OUTPUT_DIR)

    # Colab download
    from google.colab import files
    files.download("ngx_xgboost_artifacts.zip")

    print("Downloading ngx_xgboost_artifacts.zip")
    print("\nNext steps:")
    print("  1. Extract the zip")
    print("  2. Copy to your backend repo:")
    print("       xgboost_model.pkl      \u2192 models/xgboost_model.pkl")
    print("       xgb_feature_list.json  \u2192 configs/xgb_feature_list.json")
    print("       xgb_metrics.json       \u2192 reports/xgb_metrics.json")
    print("  3. git add + commit + push")
    print("  4. Redeploy backend on Render")